# SMPL Human Body → Radar Simulation

**Pipeline:**
1. Scene places an SMPL body mesh with lighting
2. Tracer ray-traces the scene → reflection points + intensities
3. Radar (Dirichlet solver) generates a full MIMO frame
4. Signal processing extracts a 3D point cloud

**Requires:** smplpytorch, mitsuba (cuda_ad_rgb variant)

In [ ]:
import sys, pathlib
repo_root = pathlib.Path.cwd().parents[0]
sys.path.insert(0, str(repo_root))

import numpy as np
import torch
import matplotlib.pyplot as plt
from witwin.radar import Material, Radar, Scene, Tracer
from witwin.radar.sigproc import process_pc, process_rd

MODEL_ROOT = repo_root / 'models' / 'smpl_models'


## 1. Radar Config (TI IWR1843)

In [ ]:
config = {
    "num_tx": 3, "num_rx": 4,
    "fc": 77e9,
    "slope": 60.012,
    "adc_samples": 256,
    "adc_start_time": 6,
    "sample_rate": 4400,
    "idle_time": 7,
    "ramp_end_time": 65,
    "chirp_per_frame": 128,
    "frame_per_second": 10,
    "num_doppler_bins": 128,
    "num_range_bins": 256,
    "num_angle_bins": 64,
    "power": 15,
    "tx_loc": [[0, 0, 0], [4, 0, 0], [2, 1, 0]],
    "rx_loc": [[-6, 0, 0], [-5, 0, 0], [-4, 0, 0], [-3, 0, 0]],
}
radar = Radar(config)

## 2. Scene: SMPL Body in Front of Radar

In [ ]:
scene = Scene()

print("Adding SMPL body to scene...")
pose = np.zeros(72)        # T-pose
shape = np.zeros(10)       # average body shape
scene.add_smpl(name="human", pose=pose, shape=shape, position=[0, -1, -3], gender='male', model_root=str(MODEL_ROOT), material=Material(eps_r=3.0))

tracer = Tracer(scene, radar, resolution=128)

## 3. Ray Trace

In [ ]:
points, intensities = tracer.trace()
print(f"{points.shape[0]} reflection points")
print(f"Z range: {points[:, 2].min():.2f} ~ {points[:, 2].max():.2f} m")

In [ ]:
image = tracer.render_image()
print(f"Image shape: {image.shape}, range: [{image.min():.4f}, {image.max():.4f}]")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image.cpu().numpy(), cmap='inferno', origin='upper')
ax.set_title('Fresnel Reflectance')
ax.axis('off')
## add a colorbar
plt.tight_layout()

In [ ]:
points.shape

## 4. Radar MIMO Frame

In [ ]:
velocity = torch.tensor([0, 0, 1], device='cuda')

def location_function(t):
    return intensities*10, points + velocity * t

frame = radar.mimo(location_function, t0=0)
print(f"Frame shape: {frame.shape}  (TX, RX, chirps, ADC)")

In [ ]:
frame.shape

In [ ]:
plt.plot(frame[0, 0, 0].cpu().numpy())

## 5. Point Cloud + Range-Doppler

In [ ]:
pc = process_pc(radar, frame)
print(f"{pc.shape[0]} detected points")

rd_mag, rd_map, ranges, velocities = process_rd(radar, frame, tx=0, rx=0)

## 6. Visualization

In [ ]:
fig = plt.figure(figsize=(15, 5))

# (a) Ray-traced reflection points
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
pts = points.cpu().numpy()
ax1.scatter(pts[:, 0], -pts[:, 2], pts[:, 1],
            c=intensities.cpu().numpy(), cmap='hot', s=1, alpha=0.5)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Depth (m)')
ax1.set_zlabel('Y (m)')
ax1.set_title(f'Ray-traced ({pts.shape[0]} pts)')
ax1.set_xlim(-1, 1)
ax1.set_ylim(0, 5)
ax1.set_zlim(-1, 1)
ax1.set_box_aspect([1, 2.5, 1])
ax1.view_init(elev=20, azim=-60)

# (b) Range-Doppler map
ax2 = fig.add_subplot(1, 3, 2)
ax2.imshow(
    rd_mag[:, :len(ranges)],
    extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
    aspect='auto', origin='lower', cmap='jet',
)
ax2.set_xlabel('Range (m)')
ax2.set_ylabel('Velocity (m/s)')
ax2.set_title('Range-Doppler Map')

# (c) Radar point cloud
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
if pc.shape[0] > 0:
    ax3.scatter(-pc[:, 0], pc[:, 1], pc[:, 2], c=pc[:, 4], cmap='hot', s=5)
ax3.set_xlabel('X (m)')
ax3.set_ylabel('Depth (m)')
ax3.set_zlabel('Y (m)')
ax3.set_title(f'Radar point cloud ({pc.shape[0]} pts)')
ax3.set_xlim(-1, 1)
ax3.set_ylim(0, 5)
ax3.set_zlim(-1, 1)
ax3.set_box_aspect([1, 2.5, 1])
ax3.view_init(elev=20, azim=-60)

plt.tight_layout()